(sec:solvation)=
# System solvation

In [1]:
import veloxchem as vlx

## Building systems

VeloxChem can be used to solvate system in preparation for molecular dynamics simulations. 

First, create a molecule object for the solute, in our example we choose deprotonated ibuprofen.

In [2]:
solute = vlx.Molecule.read_smiles("CC(C)CC1=CC=C(C=C1)C(C)C(=O)[O-]")

In [3]:
solute.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Second, generate a force field for the solute. By default RESP charges will be computed but, here, we instead make use of semi-empirical partial charges for faster execution.

In [4]:
ff_gen = vlx.MMForceFieldGenerator()

ff_gen.partial_charges = solute.get_partial_charges(solute.get_charge())

ff_gen.create_topology(solute)

* Info * Sum of partial charges is not a whole number.                                                                    
* Info * Compensating by removing 1.000e-06 from the largest charge.                                                      
                                                                                                                          
* Info * Using GAFF (v2.11) parameters.                                                                                   
         Reference: J. Wang, R. M. Wolf, J. W. Caldwell, P. A. Kollman, D. A. Case, J. Comput. Chem. 2004,
         25, 1157-1174.
                                                                                                                          
* Info * Updated bond length 13-15 (c -o ) to 0.139 nm                                                                    
* Info * Updated bond angle 14-13-15 (o -c -o ) to 119.379 deg                                                            


Third, we use of the `SolvationBuilder` class to create a simulation box with a solute padding 1.0 nm (default) consisting of SPC/E water (default). 

The solvator will run an *NPT* equilibration for 5 ps at 300 K.

In [5]:
solvator = vlx.SolvationBuilder()

solvator.solvate(solute, equilibrate=False, neutralize=False)

                                               VeloxChem Solvation Builder                                                
                                                                                                                          
* Info * Solvating the solute with cspce molecules                                                                        
* Info * Padding: 1.0 nm                                                                                                  
                                                                                                                          
* Info * The box size is: 2.78 x 2.78 x 2.78 nm^3                                                                         
* Info * The volume of the solute is: 0.23 nm^3                                                                           
* Info * The volume available for the solvent is: 21.35 nm^3                                                              
* Info * The exp

In [6]:
solvator.show_solvation_box()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Mixed solvents

Solvents can be mixed. Molecule objects are created for each type of solvent molecules. 

In [7]:
water = vlx.Molecule.read_smiles("O")
propylene_glycol = vlx.Molecule.read_smiles("CC(CO)O")

Let us consider a solvation with a 50/50 mix of number of water and propylene glycol molecules in the solvent. 

In [9]:
solvator.custom_solvate(
    solute,
    solvents=[water, propylene_glycol],
    proportion=(50, 50),
    box_size=(25, 25, 25),
)

* Info * Solvated system with 70 solvent molecules out of 70 requested                                                    
* Info * Warning: 90 attempts have been made to insert the solvent molecule                                               
* Info * Consider reducing the target density                                                                             
* Info * Warning: 90 attempts have been made to insert the solvent molecule                                               
* Info * Consider reducing the target density                                                                             
* Info * Solvated system with 70 solvent molecules out of 70 requested                                                    


In [10]:
solvator.show_solvation_box()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Free energy of solvation

In [11]:
fep_drv = vlx.SolvationFepDriver()

# Settings are here chosen for a quick execution,
# it is recommended to use the default settings
fep_drv.num_em_steps = 500
fep_drv.num_equil_steps = 500
fep_drv.num_steps = 500

# Pre-equilibrate the solvation box
solvator.perform_equilibration()

# Compute solvation free energy
fep_results = fep_drv.compute_solvation_free_energy(ff_gen, solvator)

* Info * The box size after equilibration is: 2.31 x 2.31 x 2.31 nm^3                                                     
* Info * The density of the solvent after equilibration is: 172 kg/m^3                                                    
* Info * Generating the ForceField for the solvent                                                                        
* Info * Generating the ForceField for the solvent                                                                        
* Info * solute.itp file written                                                                                          
* Info * solvent_1.itp file written                                                                                       
* Info * solvent_2.itp file written                                                                                       
* Info * system.top file written                                                                                          
* Info * system.


******* JAX 64-bit mode is now on! *******
*     JAX is now set to 64-bit mode!     *
*   This MAY cause problems with other   *
*      uses of JAX in the same code.     *
******************************************



Free energy for stage 1: -379.7875 +/- 0.6907 kJ/mol
* Info * Removing GSC potential (Stage 2)...
                                                                             
* Info * Generating systems for stage 2                                                                                   
* Info * Running lambda = 1.0, stage = 2...                                                                               
* Info * Time spent in energy minimization: 2.67 s                                                                        
* Info * Time spent in pre-equilibration: 0.36 s                                                                          
* Info * Equilibration detected at frame 411 with Neff_max = 33.32                                                        
* Info * Frames saved = 89 | No subsampling (g=2.70 <= 20)                                                                
* Info * Lambda = 1.0 completed. Elapsed time: 1.48 s                                 

In [12]:
print(f'Solvation free energy: {fep_results["free_energy"]:.3f} kJ/mol')

Solvation free energy: -310.264 kJ/mol
